# TPC-H Data Generator

Generates realistic TPC-H data at Scale Factor 0.01 (~60K lineitem rows).
Follows TPC-H spec for column distributions, naming, and referential integrity.

**Output:** One CSV per table in `./data/` with `asm_` prefix to match Sieve-GNN pipeline.

**Run this BEFORE running `Sieve_GNN.ipynb` or `ind_baseline.ipynb` if `./data/` is empty.**

In [1]:
import os
import random
import csv
from datetime import date, timedelta

random.seed(42)
OUT_DIR = "./data"
os.makedirs(OUT_DIR, exist_ok=True)

SF = 0.01  # Scale factor

# --- Distributions from TPC-H spec ---
REGIONS = ["AFRICA", "AMERICA", "ASIA", "EUROPE", "MIDDLE EAST"]
NATIONS = [
    ("ALGERIA", 0), ("ARGENTINA", 1), ("BRAZIL", 1), ("CANADA", 1),
    ("EGYPT", 0), ("ETHIOPIA", 0), ("FRANCE", 3), ("GERMANY", 3),
    ("INDIA", 2), ("INDONESIA", 2), ("IRAN", 4), ("IRAQ", 4),
    ("JAPAN", 2), ("JORDAN", 4), ("KENYA", 0), ("MOROCCO", 0),
    ("MOZAMBIQUE", 0), ("PERU", 1), ("CHINA", 2), ("ROMANIA", 3),
    ("SAUDI ARABIA", 4), ("VIETNAM", 2), ("RUSSIA", 3),
    ("UNITED KINGDOM", 3), ("UNITED STATES", 1),
]
MKTSEGMENTS = ["AUTOMOBILE", "BUILDING", "FURNITURE", "HOUSEHOLD", "MACHINERY"]
PRIORITIES = ["1-URGENT", "2-HIGH", "3-MEDIUM", "4-NOT SPECIFIED", "5-LOW"]
SHIPMODES = ["REG AIR", "AIR", "RAIL", "SHIP", "TRUCK", "MAIL", "FOB"]
SHIPINSTRUCT = ["DELIVER IN PERSON", "COLLECT COD", "NONE", "TAKE BACK RETURN"]
TYPES_1 = ["STANDARD", "SMALL", "MEDIUM", "LARGE", "ECONOMY", "PROMO"]
TYPES_2 = ["ANODIZED", "BURNISHED", "PLATED", "POLISHED", "BRUSHED"]
TYPES_3 = ["TIN", "NICKEL", "BRASS", "STEEL", "COPPER"]
CONTAINERS_1 = ["SM", "MED", "LG", "JUMBO", "WRAP"]
CONTAINERS_2 = ["CASE", "BOX", "BAG", "JAR", "PKG", "PACK", "CAN", "DRUM"]

def rand_comment(maxlen):
    words = "the of and to a in is that for it as was with be by on not he i this are or his from at which but have an had they you were their been has its".split()
    out = []
    while len(" ".join(out)) < maxlen - 10:
        out.append(random.choice(words))
    return " ".join(out)[:maxlen]

def rand_phone():
    cc = random.randint(10, 34)
    return f"{cc}-{random.randint(100,999)}-{random.randint(100,999)}-{random.randint(1000,9999)}"

def rand_date(start, end):
    delta = (end - start).days
    return start + timedelta(days=random.randint(0, delta))

def write_csv(filename, header, rows):
    path = os.path.join(OUT_DIR, filename)
    with open(path, "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(header)
        w.writerows(rows)
    print(f"  {filename}: {len(rows)} rows")

# ---- REGION ----
region_rows = []
for i, name in enumerate(REGIONS):
    region_rows.append([i, name, rand_comment(100)])
write_csv("asm_region.csv", ["r_regionkey", "r_name", "r_comment"], region_rows)

# ---- NATION ----
nation_rows = []
for i, (name, rkey) in enumerate(NATIONS):
    nation_rows.append([i, name, rkey, rand_comment(100)])
write_csv("asm_nation.csv", ["n_nationkey", "n_name", "n_regionkey", "n_comment"], nation_rows)

# ---- SUPPLIER ----
N_SUPP = max(10, int(10000 * SF))
supp_rows = []
for i in range(1, N_SUPP + 1):
    supp_rows.append([
        i, f"Supplier#{i:09d}", f"Addr{random.randint(1,9999)}",
        random.randint(0, 24), rand_phone(),
        round(random.uniform(-999.99, 9999.99), 2), rand_comment(80)
    ])
write_csv("asm_supplier.csv",
          ["s_suppkey", "s_name", "s_address", "s_nationkey", "s_phone", "s_acctbal", "s_comment"],
          supp_rows)

# ---- PART ----
N_PART = max(20, int(200000 * SF))
part_rows = []
for i in range(1, N_PART + 1):
    ptype = f"{random.choice(TYPES_1)} {random.choice(TYPES_2)} {random.choice(TYPES_3)}"
    container = f"{random.choice(CONTAINERS_1)} {random.choice(CONTAINERS_2)}"
    part_rows.append([
        i, f"Part#{i}", f"Manufacturer#{random.randint(1,5)}",
        f"Brand#{random.randint(1,5)}{random.randint(1,5)}", ptype,
        random.randint(1, 50), container,
        round(900.0 + i / 10.0, 2), rand_comment(20)
    ])
write_csv("asm_part.csv",
          ["p_partkey", "p_name", "p_mfgr", "p_brand", "p_type", "p_size", "p_container", "p_retailprice", "p_comment"],
          part_rows)

# ---- PARTSUPP ----
ps_rows = []
ps_keys = set()
for pk in range(1, N_PART + 1):
    supps = random.sample(range(1, N_SUPP + 1), min(4, N_SUPP))
    for sk in supps:
        ps_rows.append([pk, sk, random.randint(1, 9999),
                        round(random.uniform(1.0, 1000.0), 2), rand_comment(150)])
        ps_keys.add((pk, sk))
write_csv("asm_partsupp.csv",
          ["ps_partkey", "ps_suppkey", "ps_availqty", "ps_supplycost", "ps_comment"],
          ps_rows)

# ---- CUSTOMER ----
N_CUST = max(15, int(150000 * SF))
cust_rows = []
for i in range(1, N_CUST + 1):
    cust_rows.append([
        i, f"Customer#{i:09d}", f"CustAddr{random.randint(1,9999)}",
        random.randint(0, 24), rand_phone(),
        round(random.uniform(-999.99, 9999.99), 2),
        random.choice(MKTSEGMENTS), rand_comment(80)
    ])
write_csv("asm_customer.csv",
          ["c_custkey", "c_name", "c_address", "c_nationkey", "c_phone", "c_acctbal", "c_mktsegment", "c_comment"],
          cust_rows)

# ---- ORDERS ----
N_ORD = max(60, int(1500000 * SF))
ord_rows = []
START_DATE = date(1992, 1, 1)
END_DATE = date(1998, 8, 2)
for i in range(1, N_ORD + 1):
    odate = rand_date(START_DATE, END_DATE)
    ord_rows.append([
        i, random.randint(1, N_CUST),
        random.choice(["O", "F", "P"]),
        round(random.uniform(800, 500000), 2),
        odate.isoformat(),
        random.choice(PRIORITIES),
        f"Clerk#{random.randint(1, max(1, N_ORD // 100)):09d}",
        0, rand_comment(60)
    ])
write_csv("asm_orders.csv",
          ["o_orderkey", "o_custkey", "o_orderstatus", "o_totalprice",
           "o_orderdate", "o_orderpriority", "o_clerk", "o_shippriority", "o_comment"],
          ord_rows)

# ---- LINEITEM ----
li_rows = []
ps_list = list(ps_keys)
for okey in range(1, N_ORD + 1):
    n_items = random.randint(1, 7)
    for lnum in range(1, n_items + 1):
        pk, sk = random.choice(ps_list)
        shipdate = rand_date(START_DATE, END_DATE)
        commitdate = shipdate + timedelta(days=random.randint(1, 90))
        receiptdate = shipdate + timedelta(days=random.randint(1, 120))
        li_rows.append([
            okey, pk, sk, lnum,
            round(random.uniform(1, 50), 2),
            round(random.uniform(900, 105000), 2),
            round(random.uniform(0, 0.10), 2),
            round(random.uniform(0, 0.08), 2),
            random.choice(["N", "R", "A"]),
            random.choice(["O", "F"]),
            shipdate.isoformat(), commitdate.isoformat(), receiptdate.isoformat(),
            random.choice(SHIPINSTRUCT), random.choice(SHIPMODES),
            rand_comment(30)
        ])
write_csv("asm_lineitem.csv",
          ["l_orderkey", "l_partkey", "l_suppkey", "l_linenumber",
           "l_quantity", "l_extendedprice", "l_discount", "l_tax",
           "l_returnflag", "l_linestatus", "l_shipdate", "l_commitdate",
           "l_receiptdate", "l_shipinstruct", "l_shipmode", "l_comment"],
          li_rows)

print(f"\nDone! 8 tables generated in {OUT_DIR}/")
print(f"Total lineitem rows: {len(li_rows)}")

  asm_region.csv: 5 rows
  asm_nation.csv: 25 rows
  asm_supplier.csv: 100 rows
  asm_part.csv: 2000 rows
  asm_partsupp.csv: 8000 rows
  asm_customer.csv: 1500 rows
  asm_orders.csv: 15000 rows
  asm_lineitem.csv: 59836 rows

Done! 8 tables generated in ./data/
Total lineitem rows: 59836
